In [1]:
library(tidyverse)
library(dbplyr)
library(bigrquery)
library(lubridate)

bq_auth()

project_id = "yhcr-prd-bradfor-bia-core"

# create connection to database
con <- DBI::dbConnect(bigrquery::bigquery(), 
                      project = project_id)

print(paste0("Connected to : ", project_id))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘dbplyr’


The following objects are masked from ‘package:dplyr’:

    ident, sql




[1] "Connected to : yhcr-prd-bradfor-bia-core"


In [2]:
dest_data = "CB_2489.cb_NCCIS_2011_to_2019"

dest_table <- tbl(con, dest_data) |>
    select(person_id, NCCIS_ACADYR, NCCIS_MonthNo, NCCIS_Year_11_Intended_Destination, NCCIS_Current_Activity_Code, 
           NCCIS_Current_Activity_Start_Date, NCCIS_NEET_Start_Date, NCCIS_Lead_LEA_Code) |> 
    # convert to df
    collect()


In [3]:
head(dest_table)

person_id,NCCIS_ACADYR,NCCIS_MonthNo,NCCIS_Year_11_Intended_Destination,NCCIS_Current_Activity_Code,NCCIS_Current_Activity_Start_Date,NCCIS_NEET_Start_Date,NCCIS_Lead_LEA_Code
<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>
749832F625AF33E7069CC7445D5E7C0C5210D8EF0BA4D3DB806ACF5EC905799E,2010/2011,12,NA,830,2007-08-30,NA,380
EBE4BDB714011EC4573C41C3FC3EAFA79F91A10ECF37A79E1E4E304351B25630,2010/2011,3,NA,820,2008-08-06,NA,888
63D417352B7A9B345D6B76757420122583FF334EBC2F2874E3EA62029A2A06CC,2010/2011,2,NA,820,2008-02-19,NA,380
7324E1EF21451B1FE7732FEDB155B7CC0EE54052788F49D1027BF6B53A44FC91,2010/2011,9,NA,820,2008-05-13,NA,380
9CE5E3220A4154A9BD65B3BB02E15C21FF6D240B47724845E4A8F2C3F8B5EE29,2010/2011,4,130,320,2008-09-01,NA,383
A7A2D6E75DC2B62F1282690150E032A671FAF9458BC14FBC4FC4C32FEF936B4B,2012/2013,5,NA,810,2008-09-15,NA,879


In [4]:
unique(dest_table$NCCIS_ACADYR)

[1] "2010/2011" "2012/2013" "2011/2012" "2014/2015" "2013/2014" "2015/2016"
[7] "2016/2017" "2017/2018" "2018/2019"

In [11]:
dest_15_19 <- dest_table |>
    filter(NCCIS_ACADYR %in% c('2015/2016','2016/2017','2017/2018','2018/2019'))

In [7]:
dest_data_17_21 = "CB_2489.cb_NCCIS_2017_to_2021"

dest_table_17_21 <- tbl(con, dest_data_17_21) |>
    select(person_id, NCCIS_ACADYR, NCCIS_MonthNo, NCCIS_Year_11_Intended_Destination, NCCIS_Current_Activity_Code, 
           NCCIS_Current_Activity_Start_Date, NCCIS_NEET_Start_Date, NCCIS_Lead_LEA_Code) |> 
    # convert to df
    collect()


In [8]:
head(dest_table_17_21)

person_id,NCCIS_ACADYR,NCCIS_MonthNo,NCCIS_Year_11_Intended_Destination,NCCIS_Current_Activity_Code,NCCIS_Current_Activity_Start_Date,NCCIS_NEET_Start_Date,NCCIS_Lead_LEA_Code
<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>
NA,2020/2021,4,NA,820,2014-02-20,NA,888
D7CA98F8593422C2876539F75068B17F69CE4B2A5C777D3BDCE3D3D317A8F305,2016/2017,4,511,820,2014-11-11,NA,815
0EEC5BC24C4E35AF968D313FC831462BBAF3F7BCD36E8AA89AEF47EDA1AA9C0F,2016/2017,6,NA,810,2015-01-14,NA,384
1F7B068D8DE8C9860DAFB315A6C4A1BD86F681F931B39B698119D54F2EA02FFC,2020/2021,6,NA,810,2015-01-15,NA,888
9F254BBC32219DCDC43637188069FCDDD92A3BA521874552E471E9DAB9782DD9,2017/2018,9,NA,820,2015-02-23,NA,872
4B9682E08E82FBA1AD74322CDD58A793244E5E19058B37044177BFD21D832FD9,2017/2018,10,NA,820,2015-06-01,NA,909


In [9]:
dest_table_17_21 <- dest_table_17_21 |>
    filter(!is.na(person_id))

In [10]:
unique(dest_table_17_21$NCCIS_ACADYR)

[1] "2016/2017" "2020/2021" "2017/2018" "2018/2019" "2019/2020"

In [12]:
dest_16_20 <- dest_table_17_21 |>
    filter(NCCIS_ACADYR %in% c('2016/2017','2017/2018','2018/2019','2019/2020'))

#### Bind dataframes, avoiding duplicates

In [13]:
destinations <- bind_rows(dest_15_19, dest_16_20) |> 
  distinct()

In [14]:
destinations |> 
  summarise(num_unique = n_distinct(person_id))

num_unique
<int>
72639


## Filter destinations to target cohort

In [17]:
person_0002 <- read.csv("data/person_cohorts.csv", header = TRUE)

In [18]:
head(person_0002)

,person_id,birth_date,gender,CombinedEthnicity,cohort
,<chr>,<chr>,<chr>,<chr>,<chr>
1,D27CE553B8DA52E66BCBA43420F6F53DAEBF9FE2864361849339028B6B002695,2000-09-15,F,South Asian,2000/01
2,9F33F6D292805042AF92186788B71C7302C7E976F6C148C34FDE62B157C03928,2000-09-15,F,South Asian,2000/01
3,C38AD0B73560547A95C529E42A9CF6E1E06282163BB419DC2C1451422CD20CE0,2000-09-15,F,South Asian,2000/01
4,03A81FF3A58E65546A42780919E74680BF016F6D3E8FE1FA64FFD9B9E436B795,2000-09-15,F,NA,2000/01
5,8C9979BE485657CDE5AC92361D6E454DC61C1396BCC38F77933217649F802EFF,2000-09-15,F,White British,2000/01
6,00ED2E8CBC0AE48B370DBB70FB9440A00BD65113392DE33A5E14380D5C306075,2000-09-15,F,White British,2000/01


In [19]:
person_9800 = read.csv("data/additional_person_cohorts.csv", header = TRUE)

In [20]:
head(person_9800)

,person_id,birth_date,gender,CombinedEthnicity,cohort
,<chr>,<chr>,<chr>,<chr>,<chr>
1,EE91E3C77F2B7CB44F1F601317E2DE5EE76ABB0941F24D4714026B3E30FC313C,1998-09-15,M,South Asian,1998/99
2,C5553DE427E9B8F60E9B5D1E83391E1B660F415A3924B630742D52A0517F5219,1998-09-15,M,White British,1998/99
3,AC8CB4071E24E606BFFECC380A9EA11546158370916C4D8D954FA76D63FA81AE,1998-09-15,M,White British,1998/99
4,E4E84D0DEB6692712A8E875B701E097EB61235069C56626900C0951E9B112B6D,1998-09-15,M,South Asian,1998/99
5,64FA4602D88C275E27FDBD17EA4678E378D7B297D91C2CDB914150ED0CBF3785,1998-09-15,M,White British,1998/99
6,F516562D5CFDDC6CD66FB8EA18399D11BB0E2E042F569D918285FF96C3377338,1998-09-15,M,White British,1998/99


In [21]:
person <- bind_rows(person_9800, person_0002) |> 
  distinct()

In [22]:
person |> 
  summarise(num_unique = n_distinct(person_id))

num_unique
<int>
45486


In [64]:
write.csv(person, "data/person_cohorts_max.csv", row.names = FALSE)

In [23]:
destinations_filtered <- destinations |>
    # filter destinations to target person ids
    semi_join(person, by = "person_id") |>
    # then left join dfs
    left_join(person |> select(person_id, cohort), by = "person_id")

In [24]:
NEET_activity <- destinations_filtered |>
    filter(NCCIS_Current_Activity_Code %in% c('540','610','615','616','619','620','630','640','650','660','670','680','710','810','820','830'))

# ,'710','810','820','830'

In [25]:
NEET_activity  |>
 group_by(cohort) |>
 tally()

cohort,n
<chr>,<int>
1998/99,25372
1999/00,25807
2000/01,25699
2001/02,26102


In [26]:
NEET_activity |> 
  group_by(cohort) |>
  summarise(num_unique = n_distinct(person_id))

cohort,num_unique
<chr>,<int>
1998/99,6544
1999/00,6337
2000/01,6401
2001/02,6344


## Tidy destinations and activity

In [27]:
destinations_clean <- destinations_filtered |>
    mutate(Intended_destination = 
        case_when(
            NCCIS_Year_11_Intended_Destination %in% c('111','121','211','311') ~ 'EET',
            NCCIS_Year_11_Intended_Destination %in% c('411','511') ~ 'NEET',
            TRUE ~ NA
            )
        )

In [28]:
destinations_clean |>
    group_by(Intended_destination) |>
    tally()

Intended_destination,n
<chr>,<int>
EET,61429
NEET,1350
NA,777808


In [29]:
destinations_clean <- destinations_clean |>
    mutate(Current_activity = 
           case_when(
               NCCIS_Current_Activity_Code %in% c('210','220','230','240','250','260','270','280','290') ~ 'Education',
               NCCIS_Current_Activity_Code %in% c('310','320','330','340','350','360','380','381','550') ~ 'Employment',
               NCCIS_Current_Activity_Code %in% c('410','430','440','450','460') ~ 'Training',
               NCCIS_Current_Activity_Code %in% c('530','540','610','615','616','619','620','630','640','650','660','670','680','710') ~ 'NEET',
               NCCIS_Current_Activity_Code %in% c('810','820') ~ NA,
               NCCIS_Current_Activity_Code == '830' ~ 'Refused',
               TRUE ~ NA
               )
           )
               

In [30]:
destinations_clean |>
    group_by(Current_activity) |>
    tally()

Current_activity,n
<chr>,<int>
Education,639506
Employment,75156
NEET,27659
Refused,1220
Training,22483
NA,74563


In [31]:
destinations_clean <- destinations_clean |>
    mutate(NEET_detail = 
           case_when(
               NCCIS_Current_Activity_Code == '530' ~ 'Re-engagement',
               NCCIS_Current_Activity_Code == '540' ~ 'Working: no reward',
               NCCIS_Current_Activity_Code == '610' ~ 'Not ready',
               NCCIS_Current_Activity_Code == '615' ~ 'Start date agreed',
               NCCIS_Current_Activity_Code == '616' ~ 'Start date agreed',
               NCCIS_Current_Activity_Code == '619' ~ 'Seeking EET',
               NCCIS_Current_Activity_Code == '620' ~ 'Carer',
               NCCIS_Current_Activity_Code == '630' ~ 'Teen parent',
               NCCIS_Current_Activity_Code == '640' ~ 'Illness',
               NCCIS_Current_Activity_Code == '650' ~ 'Pregnancy',
               NCCIS_Current_Activity_Code == '660' ~ 'Religious grounds',
               NCCIS_Current_Activity_Code == '670' ~ 'Economically inactive',
               NCCIS_Current_Activity_Code == '680' ~ 'Other',
               NCCIS_Current_Activity_Code == '710' ~ 'Custody',
               TRUE ~ NA
               )
           )

In [32]:
destinations_clean |>
    group_by(NEET_detail) |>
    tally()

NEET_detail,n
<chr>,<int>
Carer,401
Custody,183
Economically inactive,210
Illness,2668
Not ready,1610
Other,860
Pregnancy,1004
Re-engagement,454
Religious grounds,7


In [33]:
head(destinations_clean)

person_id,NCCIS_ACADYR,NCCIS_MonthNo,NCCIS_Year_11_Intended_Destination,NCCIS_Current_Activity_Code,NCCIS_Current_Activity_Start_Date,NCCIS_NEET_Start_Date,NCCIS_Lead_LEA_Code,cohort,Intended_destination,Current_activity,NEET_detail
<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
C9DBADB7031E637DF29DB08FF5ADAE4537683FD0299F314DAA1202D5E4EAA617,2015/2016,9,NA,310,2015-07-28,NA,382,1998/99,NA,Employment,NA
174CC1E6BD3BE665ABFDCAF441CC5AEC5464AD42C33E874A1ED4B1C1BDF6246E,2015/2016,3,NA,310,2015-08-20,NA,380,1998/99,NA,Employment,NA
0C554C9C2572686990D2418E0724CD5CCF9874B9CE92E0C251E7501C6504754D,2015/2016,10,NA,310,2015-08-21,NA,353,1998/99,NA,Employment,NA
12FB003A53BF00E70DD05011BED09201A9E1BF9A2A22492047448D5D5F459AA3,2015/2016,9,NA,810,2015-09-01,NA,938,1998/99,NA,NA,NA
AF8AECA2B925A6C0C37C5B5A5421FA0E33421708126A0C567FA89DF0A2F8233B,2015/2016,9,NA,810,2015-09-01,NA,211,1998/99,NA,NA,NA
3B525035E2F6C5D41CB726E87ECF35982E71C45C84CF974EA50E9A05AB9F048D,2015/2016,11,NA,410,2015-09-02,NA,380,1998/99,NA,Training,NA


In [34]:
destinations_clean <- destinations_clean |>
    select(-c(NCCIS_Year_11_Intended_Destination, NCCIS_Current_Activity_Code, NCCIS_Current_Activity_Start_Date, NCCIS_NEET_Start_Date, NCCIS_Lead_LEA_Code, cohort))

In [35]:
head(destinations_clean)

person_id,NCCIS_ACADYR,NCCIS_MonthNo,Intended_destination,Current_activity,NEET_detail
<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>
C9DBADB7031E637DF29DB08FF5ADAE4537683FD0299F314DAA1202D5E4EAA617,2015/2016,9,NA,Employment,NA
174CC1E6BD3BE665ABFDCAF441CC5AEC5464AD42C33E874A1ED4B1C1BDF6246E,2015/2016,3,NA,Employment,NA
0C554C9C2572686990D2418E0724CD5CCF9874B9CE92E0C251E7501C6504754D,2015/2016,10,NA,Employment,NA
12FB003A53BF00E70DD05011BED09201A9E1BF9A2A22492047448D5D5F459AA3,2015/2016,9,NA,NA,NA
AF8AECA2B925A6C0C37C5B5A5421FA0E33421708126A0C567FA89DF0A2F8233B,2015/2016,9,NA,NA,NA
3B525035E2F6C5D41CB726E87ECF35982E71C45C84CF974EA50E9A05AB9F048D,2015/2016,11,NA,Training,NA


## Destinations single record

In [39]:
destinations_clean |>
  group_by(person_id, NCCIS_ACADYR, NCCIS_MonthNo) |> 
  filter(n() > 1) |> 
  summarise(count = n(), .groups = "drop") |> 
  summarise(num_persons_with_duplicates = n_distinct(person_id))

num_persons_with_duplicates
<int>
1179


In [40]:
destinations_clean <- destinations_clean |>
  arrange(person_id, NCCIS_ACADYR, NCCIS_MonthNo, Current_activity)

In [41]:
destinations_clean |>
  group_by(person_id, NCCIS_ACADYR, NCCIS_MonthNo) |> 
  filter(n() > 1)

person_id,NCCIS_ACADYR,NCCIS_MonthNo,Intended_destination,Current_activity,NEET_detail
<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>
00A6CFE5C25E69EC51F67C74E4F40F8AECFE43014AD77CBCB2ADA9ECBBB96BA6,2018/2019,10,EET,Education,NA
00A6CFE5C25E69EC51F67C74E4F40F8AECFE43014AD77CBCB2ADA9ECBBB96BA6,2018/2019,10,NA,NA,NA
00A6CFE5C25E69EC51F67C74E4F40F8AECFE43014AD77CBCB2ADA9ECBBB96BA6,2018/2019,11,EET,Education,NA
00A6CFE5C25E69EC51F67C74E4F40F8AECFE43014AD77CBCB2ADA9ECBBB96BA6,2018/2019,11,NA,NA,NA
00A6CFE5C25E69EC51F67C74E4F40F8AECFE43014AD77CBCB2ADA9ECBBB96BA6,2018/2019,12,NA,Education,NA
00A6CFE5C25E69EC51F67C74E4F40F8AECFE43014AD77CBCB2ADA9ECBBB96BA6,2018/2019,12,NA,NA,NA
00C823B541B2D9EAA0C42DE865BD315331166CFCBCC6334744ADB31E5F72D916,2016/2017,1,NA,Education,NA
00C823B541B2D9EAA0C42DE865BD315331166CFCBCC6334744ADB31E5F72D916,2016/2017,1,NA,Education,NA
00C823B541B2D9EAA0C42DE865BD315331166CFCBCC6334744ADB31E5F72D916,2016/2017,2,NA,Education,NA


#### split intended destinations and currect activity before removing 'not known' duplicates

In [42]:
dest_split <- destinations_clean |>
    select(-c(Current_activity, NEET_detail))

In [43]:
dest_split <- dest_split |>
  arrange(person_id, Intended_destination)

In [44]:
dest_split |>
  group_by(person_id) |> 
  filter(n() > 1)

person_id,NCCIS_ACADYR,NCCIS_MonthNo,Intended_destination
<chr>,<chr>,<dbl>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,1,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,2,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,3,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,4,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,5,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,6,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,7,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,8,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,9,NA


In [45]:
# order of priority: EET, NEET, NA
dest_split_clean <- dest_split |>
    group_by(person_id) |> 
    distinct(person_id, .keep_all = TRUE) |>
    select(c(person_id,Intended_destination))

In [46]:
dest_split_clean |>
    group_by(person_id) |> 
      filter(n() > 1) |> 
      summarise(count = n(), .groups = "drop") |> 
      summarise(num_persons_with_duplicates = n_distinct(person_id))

num_persons_with_duplicates
<int>
0


In [47]:
head(dest_split_clean)

person_id,Intended_destination
<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,EET
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,NA
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,NA
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,NA
000709767CB306493FC46DAC5E2E6B92D691A0C8A5299139ACA3871DAE72A909,NA


In [48]:
activ_split <- destinations_clean |>
    select(-Intended_destination)

In [49]:
# automatic order of priority: Edu, Emp, NEET, Refused, Train, NA
activ_split <- activ_split |>
    mutate(Current_activity = factor(Current_activity, levels = c("Education", "Employment", "Training", "NEET", "Refused", NA ))) |>
    arrange(person_id, NCCIS_ACADYR, NCCIS_MonthNo, Current_activity)

In [50]:
activ_split |>
  group_by(person_id, NCCIS_ACADYR, NCCIS_MonthNo) |> 
  filter(n() > 1)

person_id,NCCIS_ACADYR,NCCIS_MonthNo,Current_activity,NEET_detail
<chr>,<chr>,<dbl>,<fct>,<chr>
00A6CFE5C25E69EC51F67C74E4F40F8AECFE43014AD77CBCB2ADA9ECBBB96BA6,2018/2019,10,Education,NA
00A6CFE5C25E69EC51F67C74E4F40F8AECFE43014AD77CBCB2ADA9ECBBB96BA6,2018/2019,10,NA,NA
00A6CFE5C25E69EC51F67C74E4F40F8AECFE43014AD77CBCB2ADA9ECBBB96BA6,2018/2019,11,Education,NA
00A6CFE5C25E69EC51F67C74E4F40F8AECFE43014AD77CBCB2ADA9ECBBB96BA6,2018/2019,11,NA,NA
00A6CFE5C25E69EC51F67C74E4F40F8AECFE43014AD77CBCB2ADA9ECBBB96BA6,2018/2019,12,Education,NA
00A6CFE5C25E69EC51F67C74E4F40F8AECFE43014AD77CBCB2ADA9ECBBB96BA6,2018/2019,12,NA,NA
00C823B541B2D9EAA0C42DE865BD315331166CFCBCC6334744ADB31E5F72D916,2016/2017,1,Education,NA
00C823B541B2D9EAA0C42DE865BD315331166CFCBCC6334744ADB31E5F72D916,2016/2017,1,Education,NA
00C823B541B2D9EAA0C42DE865BD315331166CFCBCC6334744ADB31E5F72D916,2016/2017,2,Education,NA


In [51]:
# manual order of priority
activ_split_clean <- activ_split |>
    group_by(person_id, NCCIS_ACADYR, NCCIS_MonthNo) |> 
    distinct(person_id, NCCIS_ACADYR, NCCIS_MonthNo, .keep_all = TRUE)

In [52]:
activ_split_clean |>
    group_by(person_id, NCCIS_ACADYR, NCCIS_MonthNo) |> 
      filter(n() > 1) |> 
      summarise(count = n(), .groups = "drop") |> 
      summarise(num_persons_with_duplicates = n_distinct(person_id))

num_persons_with_duplicates
<int>
0


In [53]:
activ_split_clean

person_id,NCCIS_ACADYR,NCCIS_MonthNo,Current_activity,NEET_detail
<chr>,<chr>,<dbl>,<fct>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,1,Education,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,2,Education,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,3,Education,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,4,Education,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,5,Education,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,6,Education,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,7,Education,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,8,Education,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,9,Education,NA


#### Re-join and pivot wide

In [54]:
dest_joined <- activ_split_clean |>
    left_join(dest_split_clean, by = join_by(person_id)) |>
    select(c(person_id,NCCIS_ACADYR,Intended_destination,NCCIS_MonthNo,Current_activity,NEET_detail))

In [55]:
dest_joined |>
    group_by(person_id, NCCIS_ACADYR, NCCIS_MonthNo) |> 
      filter(n() > 1) |> 
      summarise(count = n(), .groups = "drop") |> 
      summarise(num_persons_with_duplicates = n_distinct(person_id))

num_persons_with_duplicates
<int>
0


In [56]:
dest_joined |>
    group_by(person_id, NCCIS_ACADYR, NCCIS_MonthNo) |> 
      filter(n() > 1)

person_id,NCCIS_ACADYR,Intended_destination,NCCIS_MonthNo,Current_activity,NEET_detail
<chr>,<chr>,<chr>,<dbl>,<fct>,<chr>


In [57]:
dest_joined 

person_id,NCCIS_ACADYR,Intended_destination,NCCIS_MonthNo,Current_activity,NEET_detail
<chr>,<chr>,<chr>,<dbl>,<fct>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,NA,1,Education,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,NA,2,Education,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,NA,3,Education,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,NA,4,Education,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,NA,5,Education,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,NA,6,Education,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,NA,7,Education,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,NA,8,Education,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,NA,9,Education,NA


In [58]:
dest_joined |>
    group_by(Current_activity) |>
    tally()

Current_activity,n
<fct>,<int>
Education,634596
Employment,74012
Training,22271
NEET,26884
Refused,1155
NA,71094


In [59]:
destinations_wide <- dest_joined |>
    select(-(NEET_detail)) |>
    pivot_wider(
        names_from = NCCIS_MonthNo,
        values_from = Current_activity,
        values_fill = NA
        ) |>
    select(c(person_id,NCCIS_ACADYR,Intended_destination,'9','10','11','12','1','2','3','4','5','6','7','8'))

In [60]:
# Save as CSV

In [61]:
write.csv(destinations_wide, "data/destinations_wide_max.csv", row.names = FALSE)

In [62]:
dest_joined_detail <- dest_joined |>
    mutate(Current_activity = case_when(
        Current_activity == "NEET" & !is.na(NEET_detail) ~ paste0("NEET: ", NEET_detail),
        TRUE ~ Current_activity
  ))

In [91]:
destinations_wide_detail <- dest_joined_detail |>
    select(-(NEET_detail)) |>
    mutate(year_month = paste(NCCIS_ACADYR, NCCIS_MonthNo, sep = ': '))|>
    ungroup() |> 
    select(-c(NCCIS_ACADYR, NCCIS_MonthNo))

In [93]:
head(destinations_wide_detail)

person_id,Intended_destination,Current_activity,year_month
<chr>,<chr>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,NA,Education,2017/2018: 1
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,NA,Education,2017/2018: 2
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,NA,Education,2017/2018: 3
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,NA,Education,2017/2018: 4
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,NA,Education,2017/2018: 5
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,NA,Education,2017/2018: 6


In [94]:
destinations_wide_detail <- destinations_wide_detail |>
    pivot_wider(
        names_from = year_month,
        values_from = Current_activity,
        values_fill = NA
        )
    #select(c(person_id,NCCIS_ACADYR,Intended_destination,'9','10','11','12','1','2','3','4','5','6','7','8'))

In [95]:
destinations_wide_detail

person_id,Intended_destination,2017/2018: 1,2017/2018: 2,2017/2018: 3,2017/2018: 4,2017/2018: 5,2017/2018: 6,2017/2018: 7,2017/2018: 8,⋯,2015/2016: 5,2015/2016: 6,2015/2016: 7,2015/2016: 8,2015/2016: 9,2015/2016: 10,2015/2016: 11,2015/2016: 12,2015/2016: 1,2015/2016: 4
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,NA,Education,Education,Education,Education,Education,Education,Education,Education,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,EET,Education,Education,Education,Education,Education,Education,Education,Education,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,NA,Education,Education,Education,Education,Education,Education,Education,Education,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,Employment,Employment,Employment,Employment,Education,Education,Education,NA,NA,NA
000709767CB306493FC46DAC5E2E6B92D691A0C8A5299139ACA3871DAE72A909,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,Education,Education,NA,NA,NA,Education,Education,Education,NA,NA
0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
00131B1E9F8FD120B0E688913FCFE2F0C27F1B2C6584AE0378B38B75E74B10AC,NA,Education,Education,Education,Education,Education,Education,Education,Education,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
00149D26484ED3BE2DEF154B8DB53A36B1A9FF0839C447A2CE19B0750225BA66,NA,Education,Education,Education,Education,Education,Education,Education,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [96]:
colnames(destinations_wide_detail)

[1] "person_id"            "Intended_destination" "2017/2018: 1"        
 [4] "2017/2018: 2"         "2017/2018: 3"         "2017/2018: 4"        
 [7] "2017/2018: 5"         "2017/2018: 6"         "2017/2018: 7"        
[10] "2017/2018: 8"         "2017/2018: 9"         "2017/2018: 10"       
[13] "2017/2018: 11"        "2017/2018: 12"        "2018/2019: 1"        
[16] "2018/2019: 2"         "2018/2019: 3"         "2018/2019: 4"        
[19] "2018/2019: 5"         "2018/2019: 6"         "2018/2019: 7"        
[22] "2018/2019: 8"         "2018/2019: 9"         "2018/2019: 10"       
[25] "2018/2019: 11"        "2018/2019: 12"        "2016/2017: 1"        
[28] "2016/2017: 2"         "2016/2017: 3"         "2016/2017: 4"        
[31] "2016/2017: 5"         "2016/2017: 6"         "2016/2017: 7"        
[34] "2016/2017: 8"         "2016/2017: 9"         "2016/2017: 10"       
[37] "2016/2017: 11"        "2016/2017: 12"        "2019/2020: 1"        
[40] "2019/2020: 2"         "2019/2020: 3"         "2019/2020: 4"        
[43] "2019/2020: 5"         "2019/2020: 6"         "2019/2020: 7"        
[46] "2019/2020: 8"         "2019/2020: 9"         "2019/2020: 10"       
[49] "2019/2020: 11"        "2019/2020: 12"        "2015/2016: 2"        
[52] "2015/2016: 3"         "2015/2016: 5"         "2015/2016: 6"        
[55] "2015/2016: 7"         "2015/2016: 8"         "2015/2016: 9"        
[58] "2015/2016: 10"        "2015/2016: 11"        "2015/2016: 12"       
[61] "2015/2016: 1"         "2015/2016: 4"

In [112]:
destinations_wide_detail_final <- destinations_wide_detail |>
    select('person_id',
            'Intended_destination',

            '2015/2016: 9',
            '2015/2016: 10',
            '2015/2016: 11',
            '2015/2016: 12',
            '2015/2016: 1',
            '2015/2016: 2',
            '2015/2016: 3',
            '2015/2016: 4',
            '2015/2016: 5',
            '2015/2016: 6',
            '2015/2016: 7',
            '2015/2016: 8',

            '2016/2017: 9',
            '2016/2017: 10',
            '2016/2017: 11',
            '2016/2017: 12',
            '2016/2017: 1',
            '2016/2017: 2',
            '2016/2017: 3',
            '2016/2017: 4',
            '2016/2017: 5',
            '2016/2017: 6',
            '2016/2017: 7',
            '2016/2017: 8',

            '2017/2018: 9',
            '2017/2018: 10',
            '2017/2018: 11',
            '2017/2018: 12',
            '2017/2018: 1',
            '2017/2018: 2',
            '2017/2018: 3',
            '2017/2018: 4',
            '2017/2018: 5',
            '2017/2018: 6',
            '2017/2018: 7',
            '2017/2018: 8',

            '2018/2019: 9',
            '2018/2019: 10',
            '2018/2019: 11',
            '2018/2019: 12',
            '2018/2019: 1',
            '2018/2019: 2',
            '2018/2019: 3',
            '2018/2019: 4',
            '2018/2019: 5',
            '2018/2019: 6',
            '2018/2019: 7',
            '2018/2019: 8',

            '2019/2020: 9',
            '2019/2020: 10',
            '2019/2020: 11',
            '2019/2020: 12',
            '2019/2020: 1',
            '2019/2020: 2',
            '2019/2020: 3',
            '2019/2020: 4',
            '2019/2020: 5',
            '2019/2020: 6',
            '2019/2020: 7',
            '2019/2020: 8')



In [113]:
head(destinations_wide_detail_final)

person_id,Intended_destination,2015/2016: 9,2015/2016: 10,2015/2016: 11,2015/2016: 12,2015/2016: 1,2015/2016: 2,2015/2016: 3,2015/2016: 4,⋯,2019/2020: 11,2019/2020: 12,2019/2020: 1,2019/2020: 2,2019/2020: 3,2019/2020: 4,2019/2020: 5,2019/2020: 6,2019/2020: 7,2019/2020: 8
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,EET,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,NA,Education,Education,Education,NA,NA,Employment,Employment,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
000709767CB306493FC46DAC5E2E6B92D691A0C8A5299139ACA3871DAE72A909,NA,NA,Education,Education,Education,NA,Education,Education,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


#### save destinations with NEET detail as csv 

In [99]:
write.csv(destinations_wide_detail_final, "data/destinations_wide_detail_max.csv", row.names = FALSE)

#### Align destination activity 2 years

In [101]:
person_cohorts <- person |>
    select(c(person_id, cohort))

In [103]:
head(person_cohorts)

,person_id,cohort
,<chr>,<chr>
1,EE91E3C77F2B7CB44F1F601317E2DE5EE76ABB0941F24D4714026B3E30FC313C,1998/99
2,C5553DE427E9B8F60E9B5D1E83391E1B660F415A3924B630742D52A0517F5219,1998/99
3,AC8CB4071E24E606BFFECC380A9EA11546158370916C4D8D954FA76D63FA81AE,1998/99
4,E4E84D0DEB6692712A8E875B701E097EB61235069C56626900C0951E9B112B6D,1998/99
5,64FA4602D88C275E27FDBD17EA4678E378D7B297D91C2CDB914150ED0CBF3785,1998/99
6,F516562D5CFDDC6CD66FB8EA18399D11BB0E2E042F569D918285FF96C3377338,1998/99


In [114]:
destinations_cohorts <- destinations_wide_detail_final |>
    left_join(person_cohorts , by = "person_id")

In [115]:
head(destinations_cohorts)

person_id,Intended_destination,2015/2016: 9,2015/2016: 10,2015/2016: 11,2015/2016: 12,2015/2016: 1,2015/2016: 2,2015/2016: 3,2015/2016: 4,⋯,2019/2020: 12,2019/2020: 1,2019/2020: 2,2019/2020: 3,2019/2020: 4,2019/2020: 5,2019/2020: 6,2019/2020: 7,2019/2020: 8,cohort
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,2000/01
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,EET,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,1999/00
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,2000/01
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,2001/02
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,NA,Education,Education,Education,NA,NA,Employment,Employment,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,1998/99
000709767CB306493FC46DAC5E2E6B92D691A0C8A5299139ACA3871DAE72A909,NA,NA,Education,Education,Education,NA,Education,Education,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,1998/99


In [116]:
destinations_cohort_9899 <- destinations_cohorts |>
    filter(cohort == '1998/99')

In [117]:
destinations_cohort_9900 <- destinations_cohorts |>
    filter(cohort == '1999/00')

In [118]:
destinations_cohort_0001 <- destinations_cohorts |>
    filter(cohort == '2000/01')

In [119]:
destinations_cohort_0102 <- destinations_cohorts |>
    filter(cohort == '2001/02')

In [129]:
destinations_cohort_9899_2years <- destinations_cohort_9899 |>
    select('cohort',
            'person_id',
            'Intended_destination',

            '2015/2016: 9',
            '2015/2016: 10',
            '2015/2016: 11',
            '2015/2016: 12',
            '2015/2016: 1',
            '2015/2016: 2',
            '2015/2016: 3',
            '2015/2016: 4',
            '2015/2016: 5',
            '2015/2016: 6',
            '2015/2016: 7',
            '2015/2016: 8',

            '2016/2017: 9',
            '2016/2017: 10',
            '2016/2017: 11',
            '2016/2017: 12',
            '2016/2017: 1',
            '2016/2017: 2',
            '2016/2017: 3',
            '2016/2017: 4',
            '2016/2017: 5',
            '2016/2017: 6',
            '2016/2017: 7',
            '2016/2017: 8')

In [130]:
destinations_cohort_9900_2years <- destinations_cohort_9900 |>
    select('cohort',
            'person_id',
            'Intended_destination',

            '2016/2017: 9',
            '2016/2017: 10',
            '2016/2017: 11',
            '2016/2017: 12',
            '2016/2017: 1',
            '2016/2017: 2',
            '2016/2017: 3',
            '2016/2017: 4',
            '2016/2017: 5',
            '2016/2017: 6',
            '2016/2017: 7',
            '2016/2017: 8',
           
           '2017/2018: 9',
            '2017/2018: 10',
            '2017/2018: 11',
            '2017/2018: 12',
            '2017/2018: 1',
            '2017/2018: 2',
            '2017/2018: 3',
            '2017/2018: 4',
            '2017/2018: 5',
            '2017/2018: 6',
            '2017/2018: 7',
            '2017/2018: 8'
          )

In [131]:
destinations_cohort_0001_2years <- destinations_cohort_0001 |>
    select('cohort',
            'person_id',
            'Intended_destination',
           
           '2017/2018: 9',
            '2017/2018: 10',
            '2017/2018: 11',
            '2017/2018: 12',
            '2017/2018: 1',
            '2017/2018: 2',
            '2017/2018: 3',
            '2017/2018: 4',
            '2017/2018: 5',
            '2017/2018: 6',
            '2017/2018: 7',
            '2017/2018: 8',
           
           '2018/2019: 9',
            '2018/2019: 10',
            '2018/2019: 11',
            '2018/2019: 12',
            '2018/2019: 1',
            '2018/2019: 2',
            '2018/2019: 3',
            '2018/2019: 4',
            '2018/2019: 5',
            '2018/2019: 6',
            '2018/2019: 7',
            '2018/2019: 8'
          )

In [132]:
destinations_cohort_0102_2years <- destinations_cohort_0102 |>
    select('cohort',
            'person_id',
            'Intended_destination',
           
           '2018/2019: 9',
            '2018/2019: 10',
            '2018/2019: 11',
            '2018/2019: 12',
            '2018/2019: 1',
            '2018/2019: 2',
            '2018/2019: 3',
            '2018/2019: 4',
            '2018/2019: 5',
            '2018/2019: 6',
            '2018/2019: 7',
            '2018/2019: 8',
           
           '2019/2020: 9',
            '2019/2020: 10',
            '2019/2020: 11',
            '2019/2020: 12',
            '2019/2020: 1',
            '2019/2020: 2',
            '2019/2020: 3',
            '2019/2020: 4',
            '2019/2020: 5',
            '2019/2020: 6',
            '2019/2020: 7',
            '2019/2020: 8'
          )

In [133]:
new_names <- c("Year1Month1", "Year1Month2", "Year1Month3",
              "Year1Month4", "Year1Month5", "Year1Month6",
              "Year1Month7", "Year1Month8", "Year1Month9",
              "Year1Month10", "Year1Month11", "Year1Month12",
              "Year2Month1", "Year2Month2", "Year2Month3",
              "Year2Month4", "Year2Month5", "Year2Month6",
              "Year2Month7", "Year2Month8", "Year2Month9",
              "Year2Month10", "Year2Month11", "Year2Month12")

In [134]:
colnames(destinations_cohort_9899_2years)[4:27] <- new_names
colnames(destinations_cohort_9900_2years)[4:27] <- new_names
colnames(destinations_cohort_0001_2years)[4:27] <- new_names
colnames(destinations_cohort_0102_2years)[4:27] <- new_names

In [135]:
head(destinations_cohort_0102_2years)

cohort,person_id,Intended_destination,Year1Month1,Year1Month2,Year1Month3,Year1Month4,Year1Month5,Year1Month6,Year1Month7,⋯,Year2Month3,Year2Month4,Year2Month5,Year2Month6,Year2Month7,Year2Month8,Year2Month9,Year2Month10,Year2Month11,Year2Month12
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
2001/02,0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,NA,Education,Education,Education,Education,Education,Education,NEET: Not ready,⋯,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET
2001/02,0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,NA,Education,Education,Education,Education,Education,Education,Education,⋯,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education
2001/02,0018E26A40A6299391164D4918658B0B1248367C762DCE6D2FDF1AE131A9138C,NA,NA,Education,Education,Education,Education,Education,Education,⋯,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education
2001/02,0019031CD62586D59268772800516ED50CDF1EF7DE46488542BB031850F6E68A,NA,NA,Education,Education,Education,Education,Education,Education,⋯,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment
2001/02,001E5C1FFA964716BF5E84562BC0B5D64F7742B1E8126718A5C0157D485F215D,NA,NA,Education,Education,Education,Education,Education,Education,⋯,Education,Education,Education,Education,Education,Education,Education,Education,Education,NA
2001/02,001F566795F57F5D8DD92430D868A38B757699844E3E09569B33253527012932,NA,NA,Education,Education,Education,Education,Education,Education,⋯,Education,Education,Education,Education,Education,Education,Education,Education,Education,NA


In [136]:
destinations_cohorts_aligned <- rbind(destinations_cohort_9899_2years,
                                      destinations_cohort_9900_2years,
                                      destinations_cohort_0001_2years,
                                      destinations_cohort_0102_2years)

In [137]:
destinations_cohorts_aligned

cohort,person_id,Intended_destination,Year1Month1,Year1Month2,Year1Month3,Year1Month4,Year1Month5,Year1Month6,Year1Month7,⋯,Year2Month3,Year2Month4,Year2Month5,Year2Month6,Year2Month7,Year2Month8,Year2Month9,Year2Month10,Year2Month11,Year2Month12
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1998/99,00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,NA,Education,Education,Education,NA,NA,Employment,Employment,⋯,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment
1998/99,000709767CB306493FC46DAC5E2E6B92D691A0C8A5299139ACA3871DAE72A909,NA,NA,Education,Education,Education,NA,Education,Education,⋯,Education,Education,Education,Education,Education,Education,Education,Education,Education,NA
1998/99,001F42538A9E6E496F13CD05B5E9FE91EB3B20F0CEBECA1B76CC5E7151B7A0D4,NA,NA,Education,NA,NA,Education,NA,Education,⋯,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education
1998/99,00211975518FD5336E43850D59B5F5DDA7CD98B212760111CB40528BDECF3B1B,NA,NA,Employment,Employment,Employment,Employment,Employment,Employment,⋯,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment
1998/99,00237E9C21696BB29409630166F2D9CCEC24B83FB120AEDD878DDD54C6516B3D,EET,NA,NA,NA,NA,NA,NA,Education,⋯,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education
1998/99,00247728E0E765C3F36CF7DBDF2A3509DB16E5CED7E4707969748F4B62DA78A9,NA,Education,Education,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,⋯,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment
1998/99,0025102C0D8E8C5F27454E3715E982A43E1431991E85562E55BCB19D9D1BD7D3,NA,Education,Education,Education,Education,Education,Education,Education,⋯,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education
1998/99,0026BFA5D5426F421200AB505AAD52956621DCC253241A6647302BCE4764547D,NA,NA,Education,Education,Education,Education,Education,Education,⋯,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education
1998/99,002D3DB2625339C740EB129CCC925A9D90E4F5AA05561B81FE166EDF990B8F65,NA,NA,Education,Education,Education,Education,Education,Education,⋯,Education,Education,Education,Education,Education,Education,Education,Education,Education,NA


#### save destinations with NEET detail and aligned months as csv 

In [138]:
write.csv(destinations_cohorts_aligned, "data/destinations_4cohorts_aligned.csv", row.names = FALSE)